In [ ]:
#Expanded mitochondrial spot count/micron2


import os
import pandas as pd
import numpy as np
import tifffile as tiff
from skimage import filters, morphology
import bigfish.detection as detection
import napari

def analyze_mitochondrial_rna_multiotsu_napari(image_path, pixel_size_nm, spot_radius_nm, expansion_factor):
    """
    Segments mitochondrial area using Multi-Otsu, counts MTCO3 msFISH spots using Big-FISH, 
    returns fully scaled coordinates, and opens an interactive Napari viewer.
    """
    
    # 1. Load the Image
    print(f"Loading image from: {image_path}")
    img_stack = tiff.imread(image_path)
    
    if img_stack.ndim == 3 and img_stack.shape[0] >= 4:
        img_fish = img_stack[0]       # Channel 1: MTCO3 msFISH
        img_mito = img_stack[1]       # Channel 2: Mitotracker
    else:
        print("Error: Expected a multi-channel 2D image with at least 4 channels (C, Y, X).")
        return None

    # ==========================================
    # 2. MITOCHONDRIAL SEGMENTATION (Multi-Otsu)
    # ==========================================
    print("Segmenting mitochondria using Multi-Otsu...")
    thresholds_mito = filters.threshold_multiotsu(img_mito, classes=3)
    
    # We use thresholds_mito[0] to capture both the medium (edges) and bright (cores) signals.
    # If the mask is picking up too much background haze, change this to thresholds_mito[1]
    mask_mito = img_mito > thresholds_mito[0]
    
    mask_mito = morphology.remove_small_objects(mask_mito, max_size=49)
    mask_mito = morphology.remove_small_holes(mask_mito, max_size=49)
    
    pixel_area_um2 = (pixel_size_nm / 1000)**2
    total_mito_area_um2 = np.sum(mask_mito) * pixel_area_um2 / (expansion_factor**2)

    # ==========================================
    # 3. smFISH SPOT DETECTION (Big-FISH)
    # ==========================================
    print("Detecting MTCO3 spots using Big-FISH...")
    spots, threshold = detection.detect_spots(
        images=img_fish, 
        return_threshold=True, 
        voxel_size=(pixel_size_nm, pixel_size_nm),  
        spot_radius=(spot_radius_nm, spot_radius_nm) 
    )
    
    spot_coords = spots 
    total_spots_detected = len(spot_coords)
    print(f"Big-FISH auto-threshold selected: {threshold}")

    # ==========================================
    # 4. SPATIAL FILTERING
    # ==========================================
    spots_in_mito = []
    for y, x in spot_coords:
        if 0 <= y < mask_mito.shape[0] and 0 <= x < mask_mito.shape[1]:
            if mask_mito[y, x]: 
                spots_in_mito.append([y, x])
                
    spots_in_mito = np.array(spots_in_mito)
    num_spots_in_mito = len(spots_in_mito)
    
    spots_per_um2 = num_spots_in_mito / total_mito_area_um2 if total_mito_area_um2 > 0 else 0

    # ==========================================
    # 5. COORDINATE SCALING (Expanded -> Biological)
    # ==========================================
    biological_pixels_all = spot_coords / expansion_factor
    biological_pixels_mito = spots_in_mito / expansion_factor if num_spots_in_mito > 0 else np.array([])
    biological_nm_all = biological_pixels_all * pixel_size_nm
    biological_nm_mito = biological_pixels_mito * pixel_size_nm if num_spots_in_mito > 0 else np.array([])

    # ==========================================
    # 6. VISUALIZATION (NAPARI)
    # ==========================================
    print("Opening Napari viewer...")
    viewer = napari.Viewer()
    
    # Add Base Images
    viewer.add_image(img_mito, name='Mitotracker', colormap='magenta', blending='additive')
    viewer.add_image(img_fish, name='MTCO3 msFISH', colormap='gray', blending='additive')
    
    # Add Mitochondrial Mask
    viewer.add_labels(mask_mito.astype(int), name='Mito Mask (Multi-Otsu)', opacity=0.2)
    
    # Add Detected Spots
    viewer.add_points(
        spot_coords, 
        name='Detected MTCO3 Spots', 
        size=15, 
        face_color='transparent', 
        border_color='yellow',  
        border_width=0.15       
    )
    
    viewer.layers['Mitotracker'].contrast_limits = [np.percentile(img_mito, 1), np.percentile(img_mito, 99.5)]
    viewer.layers['MTCO3 msFISH'].contrast_limits = [np.percentile(img_fish, 1), np.percentile(img_fish, 99.5)]

    napari.run()
    
    return {
        "Total_Mito_Area_um2": total_mito_area_um2,
        "Total_Spots_Image": total_spots_detected,
        "Spots_in_Mito": num_spots_in_mito,
        "Density_Spots_per_um2": spots_per_um2,
        "Biological_Pixels_All": biological_pixels_all,
        "Biological_Pixels_Mito": biological_pixels_mito,
        "Biological_Nanometers_All": biological_nm_all,
        "Biological_Nanometers_Mito": biological_nm_mito
    }

# ==========================================
# EXECUTION BLOCK (WITH SAVING)
# ==========================================
file_path = '/Users/parahazr/Documents/Mito_RNAs/UREX_Mito/5.tif'

results = analyze_mitochondrial_rna_multiotsu_napari(
    image_path=file_path, 
    pixel_size_nm=58.7, 
    spot_radius_nm=200, 
    expansion_factor=5.0
)

if results:
    print("\n--- Final Metrics ---")
    print(f"Area (um2): {results['Total_Mito_Area_um2']:.2f}")
    print(f"Total Spots: {results['Total_Spots_Image']}")
    print(f"Spots in Mito: {results['Spots_in_Mito']}")
    print(f"Density (spots/um2): {results['Density_Spots_per_um2']:.2f}")

    # --- SAVE TO CSV ---
    save_dir = '/Users/parahazr/Documents/Mito_RNAs/UREX_Mito/10.tif'
    
    # 1. Save the Summary Metrics
    summary_path = os.path.join(save_dir, '11_summary_metrics.csv')
    summary_df = pd.DataFrame([{
        "Image_Name": "11.tif",
        "Total_Mito_Area_um2": results['Total_Mito_Area_um2'],
        "Total_Spots_Detected": results['Total_Spots_Image'],
        "Spots_Inside_Mito": results['Spots_in_Mito'],
        "Density_Spots_per_um2": results['Density_Spots_per_um2']
    }])
    summary_df.to_csv(summary_path, index=False)
    print(f"\n✅ Summary metrics saved to: {summary_path}")

    # 2. Save the True Biological Coordinates (Nanometers)
    coords_path = os.path.join(save_dir, '11_mito_spot_coordinates_nm.csv')
    if len(results['Biological_Nanometers_Mito']) > 0:
        coords_df = pd.DataFrame(
            results['Biological_Nanometers_Mito'], 
            columns=['Y_Coordinate_nm', 'X_Coordinate_nm']
        )
        coords_df.to_csv(coords_path, index=False)
        print(f"✅ Biological coordinates saved to: {coords_path}")
    else:
        print("No spots found inside mitochondria to save.")

In [ ]:
#Unexpanded mitochondrial spot count/micron2

import os
import pandas as pd
import numpy as np
import tifffile as tiff
from skimage import filters, morphology
import bigfish.detection as detection
import napari

def analyze_unexpanded_mitochondrial_rna_napari(image_path, pixel_size_nm, spot_radius_nm, expansion_factor=1.0):
    """
    Segments mitochondrial area (Ch 0), counts MTCO3 msFISH spots (Ch 1), 
    returns coordinates, and opens an interactive Napari viewer.
    """
    
    # 1. Load the Image
    print(f"Loading image from: {image_path}")
    img_stack = tiff.imread(image_path)
    
    # UPDATED: Check for at least 3 channels, and swap the channel mapping
    if img_stack.ndim == 3 and img_stack.shape[0] >= 3:
        img_mito = img_stack[0]       # Channel 0: Mitotracker
        img_fish = img_stack[1]       # Channel 1: MTCO3 msFISH
    else:
        print("Error: Expected a multi-channel 2D image with at least 3 channels (C, Y, X).")
        return None

    # ==========================================
    # 2. MITOCHONDRIAL SEGMENTATION (Multi-Otsu)
    # ==========================================
    print("Segmenting mitochondria using Multi-Otsu...")
    thresholds_mito = filters.threshold_multiotsu(img_mito, classes=3)
    
    mask_mito = img_mito > thresholds_mito[0]
    
    mask_mito = morphology.remove_small_objects(mask_mito, max_size=49)
    mask_mito = morphology.remove_small_holes(mask_mito, max_size=49)
    
    pixel_area_um2 = (pixel_size_nm / 1000)**2
    # Since expansion_factor is 1.0, this calculates standard biological area
    total_mito_area_um2 = np.sum(mask_mito) * pixel_area_um2 / (expansion_factor**2)

    # ==========================================
    # 3. smFISH SPOT DETECTION (Big-FISH)
    # ==========================================
    print("Detecting MTCO3 spots using Big-FISH...")
    spots, threshold = detection.detect_spots(
        images=img_fish, 
        return_threshold=True, 
        voxel_size=(pixel_size_nm, pixel_size_nm),  
        spot_radius=(spot_radius_nm, spot_radius_nm) 
    )
    
    spot_coords = spots 
    total_spots_detected = len(spot_coords)
    print(f"Big-FISH auto-threshold selected: {threshold}")

    # ==========================================
    # 4. SPATIAL FILTERING
    # ==========================================
    spots_in_mito = []
    for y, x in spot_coords:
        if 0 <= y < mask_mito.shape[0] and 0 <= x < mask_mito.shape[1]:
            if mask_mito[y, x]: 
                spots_in_mito.append([y, x])
                
    spots_in_mito = np.array(spots_in_mito)
    num_spots_in_mito = len(spots_in_mito)
    
    spots_per_um2 = num_spots_in_mito / total_mito_area_um2 if total_mito_area_um2 > 0 else 0

    # ==========================================
    # 5. COORDINATE SCALING 
    # ==========================================
    # Dividing by 1.0 keeps pixels the same, which is correct for unexpanded
    biological_pixels_all = spot_coords / expansion_factor
    biological_pixels_mito = spots_in_mito / expansion_factor if num_spots_in_mito > 0 else np.array([])
    biological_nm_all = biological_pixels_all * pixel_size_nm
    biological_nm_mito = biological_pixels_mito * pixel_size_nm if num_spots_in_mito > 0 else np.array([])

    # ==========================================
    # 6. VISUALIZATION (NAPARI)
    # ==========================================
    print("Opening Napari viewer...")
    viewer = napari.Viewer()
    
    viewer.add_image(img_mito, name='Mitotracker', colormap='magenta', blending='additive')
    viewer.add_image(img_fish, name='MTCO3 msFISH', colormap='gray', blending='additive')
    
    viewer.add_labels(mask_mito.astype(int), name='Mito Mask (Multi-Otsu)', opacity=0.2)
    
    viewer.add_points(
        spot_coords, 
        name='Detected MTCO3 Spots', 
        size=15, 
        face_color='transparent', 
        border_color='yellow',  
        border_width=0.15       
    )
    
    viewer.layers['Mitotracker'].contrast_limits = [np.percentile(img_mito, 1), np.percentile(img_mito, 99.5)]
    viewer.layers['MTCO3 msFISH'].contrast_limits = [np.percentile(img_fish, 1), np.percentile(img_fish, 99.5)]

    napari.run()
    
    return {
        "Total_Mito_Area_um2": total_mito_area_um2,
        "Total_Spots_Image": total_spots_detected,
        "Spots_in_Mito": num_spots_in_mito,
        "Density_Spots_per_um2": spots_per_um2,
        "Biological_Pixels_All": biological_pixels_all,
        "Biological_Pixels_Mito": biological_pixels_mito,
        "Biological_Nanometers_All": biological_nm_all,
        "Biological_Nanometers_Mito": biological_nm_mito
    }

# ==========================================
# EXECUTION BLOCK (WITH SAVING)
# ==========================================
file_path = '/Users/parahazr/Documents/Mito_RNAs/9.tif'

results = analyze_unexpanded_mitochondrial_rna_napari(
    image_path=file_path, 
    pixel_size_nm=58.7, 
    spot_radius_nm=150,    # Lowered to 150nm for unexpanded standard FISH spots
    expansion_factor=1.0   # Set to 1.0
)

if results:
    print("\n--- Final Metrics ---")
    print(f"Area (um2): {results['Total_Mito_Area_um2']:.2f}")
    print(f"Total Spots: {results['Total_Spots_Image']}")
    print(f"Spots in Mito: {results['Spots_in_Mito']}")
    print(f"Density (spots/um2): {results['Density_Spots_per_um2']:.2f}")

    # --- SAVE TO CSV ---
    save_dir = '/Users/parahazr/Documents/Mito_RNAs/'
    
    # 1. Save the Summary Metrics (Added 'unexpanded' to filename to prevent overwriting)
    summary_path = os.path.join(save_dir, '9_unexpanded_summary_metrics.csv')
    summary_df = pd.DataFrame([{
        "Image_Name": "9.tif",
        "Total_Mito_Area_um2": results['Total_Mito_Area_um2'],
        "Total_Spots_Detected": results['Total_Spots_Image'],
        "Spots_Inside_Mito": results['Spots_in_Mito'],
        "Density_Spots_per_um2": results['Density_Spots_per_um2']
    }])
    summary_df.to_csv(summary_path, index=False)
    print(f"\n✅ Summary metrics saved to: {summary_path}")

    # 2. Save the True Biological Coordinates (Nanometers)
    coords_path = os.path.join(save_dir, '9_unexpanded_mito_spot_coordinates_nm.csv')
    if len(results['Biological_Nanometers_Mito']) > 0:
        coords_df = pd.DataFrame(
            results['Biological_Nanometers_Mito'], 
            columns=['Y_Coordinate_nm', 'X_Coordinate_nm']
        )
        coords_df.to_csv(coords_path, index=False)
        print(f"✅ Biological coordinates saved to: {coords_path}")
    else:
        print("No spots found inside mitochondria to save.")

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import tifffile as tiff
from skimage import filters, measure, morphology
from skimage.feature import peak_local_max
from scipy.spatial import distance

def get_pixel_size_nm(image_path, fallback_nm=18.08):
    """Extracts physical pixel size from metadata, handling ImageJ/Leica custom units."""
    try:
        with tiff.TiffFile(image_path) as tif:
            page = tif.pages[0]
            if 'XResolution' in page.tags:
                x_res_tag = page.tags['XResolution'].value
                unit_tag = page.tags.get('ResolutionUnit').value if 'ResolutionUnit' in page.tags else 1
                
                if isinstance(x_res_tag, tuple) and len(x_res_tag) == 2:
                    pixels_per_unit = x_res_tag[0] / x_res_tag[1] if x_res_tag[1] != 0 else 0
                else:
                    pixels_per_unit = float(x_res_tag)
                    
                if pixels_per_unit > 0:
                    if unit_tag == 1:
                        physical_unit = 'micron' 
                        if tif.is_imagej and tif.imagej_metadata:
                            physical_unit = tif.imagej_metadata.get('unit', 'micron').lower()
                        if physical_unit in ['um', 'micron', 'microns', 'µm']:
                            return (1 / pixels_per_unit) * 1000.0 
                        elif physical_unit == 'nm':
                            return (1 / pixels_per_unit)
                    elif unit_tag == 3: 
                        return (1 / pixels_per_unit) * 1e7 
                    elif unit_tag == 2: 
                        return (1 / pixels_per_unit) * 25.4 * 1e6 
    except Exception as e:
        pass
    return fallback_nm


def get_otsu_mask(image):
    """Creates a clean mask to ignore background noise."""
    try:
        thresholds = filters.threshold_multiotsu(image, classes=3)
        mask = image > thresholds[1]
        mask = morphology.remove_small_objects(mask, max_size=4)
        return mask
    except ValueError:
        return image > filters.threshold_otsu(image)


def analyze_tri_channel_architecture(image_path, pixel_size_nm):
    img_stack = tiff.imread(image_path)
    
    # 0: 5' (Atto 633), 1: 3' (Atto 565), 2: Mid (Atto 488)
    if img_stack.ndim >= 3 and img_stack.shape[0] >= 3:
        img_5p = img_stack[0]   
        img_3p = img_stack[1]   
        img_mid = img_stack[2]  
    else:
        print(f"  Skipping {os.path.basename(image_path)}: Needs at least 3 channels.")
        return []

    # 1. Masking
    mask_5p = get_otsu_mask(img_5p)
    mask_mid = get_otsu_mask(img_mid)
    mask_3p = get_otsu_mask(img_3p)
    
    # 2. Extract Coordinates (Multiple Peaks for Shells, Multiple Centroids for Cores)
    peaks_5p = peak_local_max(img_5p, min_distance=3, labels=mask_5p)
    peaks_3p = peak_local_max(img_3p, min_distance=3, labels=mask_3p)
    
    labels_mid = measure.label(mask_mid)
    # Get centroids for EVERY distinct Mid core found in the image
    centroids_mid = [prop.centroid for prop in measure.regionprops(labels_mid)]
    
    if len(peaks_5p) == 0 or len(peaks_3p) == 0 or len(centroids_mid) == 0:
        return []

    file_name = os.path.basename(image_path)
    image_results = []

    # --- CALCULATION A: 5' to Nearest Mid Core ---
    dist_matrix_5p_mid = distance.cdist(peaks_5p, centroids_mid, metric='euclidean')
    min_dist_5p_mid = dist_matrix_5p_mid.min(axis=1) * pixel_size_nm
    
    for i, dist in enumerate(min_dist_5p_mid):
        image_results.append({
            "Image_Name": file_name,
            "Pixel_Size_Used_nm": round(pixel_size_nm, 2),
            "Interaction_Type": "5_Prime_to_Mid",
            "Spot_ID": i + 1,
            "Distance_nm": dist
        })

    # --- CALCULATION B: 3' to Nearest Mid Core ---
    dist_matrix_3p_mid = distance.cdist(peaks_3p, centroids_mid, metric='euclidean')
    min_dist_3p_mid = dist_matrix_3p_mid.min(axis=1) * pixel_size_nm
    
    for i, dist in enumerate(min_dist_3p_mid):
        image_results.append({
            "Image_Name": file_name,
            "Pixel_Size_Used_nm": round(pixel_size_nm, 2),
            "Interaction_Type": "3_Prime_to_Mid",
            "Spot_ID": i + 1,
            "Distance_nm": dist
        })

    # --- CALCULATION C: 5' to Nearest 3' (Looping Check) ---
    dist_matrix_5p_3p = distance.cdist(peaks_5p, peaks_3p, metric='euclidean')
    min_dist_5p_3p = dist_matrix_5p_3p.min(axis=1) * pixel_size_nm
    
    for i, dist in enumerate(min_dist_5p_3p):
        image_results.append({
            "Image_Name": file_name,
            "Pixel_Size_Used_nm": round(pixel_size_nm, 2),
            "Interaction_Type": "5_Prime_to_3_Prime",
            "Spot_ID": i + 1,
            "Distance_nm": dist
        })
        
    return image_results


# ==========================================
# EXECUTION BLOCK
# ==========================================
input_dir = '/Users/parahazr/Documents/Neat1_UREX/5_633_3_565_mid_488_UREX/together' 
output_csv = os.path.join(input_dir, 'compiled_TriChannel_Architecture.csv')

image_files = glob.glob(os.path.join(input_dir, '*.tif')) + glob.glob(os.path.join(input_dir, '*.tiff'))

print(f"Found {len(image_files)} images. Starting Nearest Neighbor 3-Channel analysis...\n")

all_distances = []

for idx, img_path in enumerate(image_files):
    current_pixel_size = get_pixel_size_nm(img_path, fallback_nm=18.08)
    print(f"Processing ({idx+1}/{len(image_files)}): {os.path.basename(img_path)} [Pixel size: {current_pixel_size:.2f} nm]")
    
    results = analyze_tri_channel_architecture(img_path, pixel_size_nm=current_pixel_size)
    all_distances.extend(results)

if all_distances:
    df = pd.DataFrame(all_distances)
    df.to_csv(output_csv, index=False)
    
    print("\n✅ Batch processing complete!")
    print(f"Data saved to: {output_csv}")
    
    print("\n--- Summary Statistics (Medians) ---")
    summary = df.groupby('Interaction_Type')['Distance_nm'].median()
    print(summary)
else:
    print("\n⚠️ No distances could be calculated.")